# Time Series Forecasting Analysis

This notebook analyzes the `student_dataset.csv` file containing hourly time series data of people counts from a sensor in Australia. The dataset has 1439 missing observations at the end that we will forecast using three different approaches:

1. **ARIMA** (AutoRegressive Integrated Moving Average)
2. **UCM** (Unobserved Components Model)
3. **Machine Learning** (Gradient Boosting with engineered features)

The historical data spans from April 2015 to December 2019, and we need to forecast from January 2020 to February 29, 2020.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from datetime import datetime, timedelta

# Forecasting models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.structural import UnobservedComponents
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 6)

## 2. Data Loading

Load the CSV file, parse the time column as datetime, and set it as the index.

In [ ]:
# Load the dataset
df = pd.read_csv('student_dataset.csv', parse_dates=['time'])
df.set_index('time', inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"\nDate range: {df.index.min()} to {df.index.max()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nLast few rows:")
print(df.tail())

## 3. Preprocessing

Identify the split between training data (observed values) and test/forecast period (missing values).

In [ ]:
# Identify training and forecasting periods
train_data = df[df['value'].notna()].copy()
forecast_period = df[df['value'].isna()].copy()

print(f"Training data: {len(train_data)} observations")
print(f"Training period: {train_data.index.min()} to {train_data.index.max()}")
print(f"\nForecast period: {len(forecast_period)} observations")
print(f"Forecast period: {forecast_period.index.min()} to {forecast_period.index.max()}")

# Basic statistics
print(f"\nTraining data statistics:")
print(train_data['value'].describe())

## 4. Visualization

Plot the historical time series to understand patterns and trends.

In [ ]:
# Plot the entire historical time series
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data['value'], linewidth=0.5, alpha=0.8)
plt.title('Historical Time Series: People Count (Hourly)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('People Count', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot the last 3 months for better visibility
last_3_months = train_data.last('90D')
plt.figure(figsize=(15, 6))
plt.plot(last_3_months.index, last_3_months['value'], linewidth=1, alpha=0.8)
plt.title('Last 3 Months of Historical Data', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('People Count', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Model 1: ARIMA

Use ARIMA model from statsmodels to forecast the missing period. We'll use ARIMA(5,1,2) as a starting point.

In [ ]:
print("Training ARIMA model...")

# Fit ARIMA model
# Using order (5,1,2) as a reasonable starting point for hourly data
arima_model = ARIMA(train_data['value'], order=(5, 1, 2))
arima_fit = arima_model.fit()

print("ARIMA model trained successfully!")
print(f"\nModel summary:")
print(arima_fit.summary())

In [ ]:
# Generate ARIMA forecasts
n_forecast = len(forecast_period)
arima_forecast = arima_fit.forecast(steps=n_forecast)
arima_forecast_df = pd.DataFrame({
    'ARIMA': arima_forecast
}, index=forecast_period.index)

print(f"ARIMA forecast generated for {n_forecast} time steps")
print(f"\nFirst 10 forecasts:")
print(arima_forecast_df.head(10))
print(f"\nForecast statistics:")
print(arima_forecast_df.describe())

## 6. Model 2: UCM (Unobserved Components Model)

Use Unobserved Components Model to capture trend and seasonal patterns.

In [ ]:
print("Training UCM model...")

# Fit UCM model with trend and seasonal components
# level='local linear trend' captures changing trends
# seasonal=24 for daily seasonality in hourly data
ucm_model = UnobservedComponents(
    train_data['value'],
    level='local linear trend',
    seasonal=24,
    stochastic_seasonal=True
)
ucm_fit = ucm_model.fit(disp=False)

print("UCM model trained successfully!")
print(f"\nModel summary:")
print(ucm_fit.summary())

In [ ]:
# Generate UCM forecasts
ucm_forecast = ucm_fit.forecast(steps=n_forecast)
ucm_forecast_df = pd.DataFrame({
    'UCM': ucm_forecast
}, index=forecast_period.index)

print(f"UCM forecast generated for {n_forecast} time steps")
print(f"\nFirst 10 forecasts:")
print(ucm_forecast_df.head(10))
print(f"\nForecast statistics:")
print(ucm_forecast_df.describe())

## 7. Model 3: Machine Learning (Gradient Boosting)

Use Gradient Boosting with engineered features from the datetime index and lag features.

In [ ]:
def create_features(data, lags=None):
    """
    Create features from datetime index and lag features.
    
    Args:
        data: DataFrame with datetime index
        lags: List of lag periods to create (1=1h, 24=1day, 168=1week, 720=30days)
    """
    if lags is None:
        lags = [1, 24, 168, 720]
    df = data.copy()
    
    # Datetime features
    df['hour'] = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['day'] = df.index.day
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    
    # Cyclical encoding for hour (24-hour cycle)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Cyclical encoding for day of week (7-day cycle)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # Lag features
    for lag in lags:
        df[f'lag_{lag}'] = df['value'].shift(lag)
    
    # Rolling statistics
    df['rolling_mean_24'] = df['value'].rolling(window=24, min_periods=1).mean()
    df['rolling_std_24'] = df['value'].rolling(window=24, min_periods=1).std()
    df['rolling_mean_168'] = df['value'].rolling(window=168, min_periods=1).mean()
    
    return df

# Create features for training data
train_with_features = create_features(train_data)

# Drop rows with NaN values from lag features
train_with_features = train_with_features.dropna()

print(f"Training data with features: {train_with_features.shape}")
print(f"\nFeature columns:")
print([col for col in train_with_features.columns if col != 'value'])

In [ ]:
# Prepare training data
feature_cols = [col for col in train_with_features.columns if col != 'value']
X_train = train_with_features[feature_cols]
y_train = train_with_features['value']

# Train Gradient Boosting model
print("Training Gradient Boosting model...")
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
gb_model.fit(X_train, y_train)

print("Gradient Boosting model trained successfully!")
print(f"\nFeature importances:")
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance.head(10))

In [ ]:
# Generate forecasts iteratively (since we need lag features)
print("Generating ML forecasts...")

# Combine train and forecast data for feature creation
full_data = pd.concat([train_data, forecast_period])
full_with_features = create_features(full_data)

# For forecasting, we need to predict iteratively
ml_forecasts = []
forecast_data = full_data.copy()

for i, idx in enumerate(forecast_period.index):
    # Create features for current timestep
    features_at_idx = create_features(forecast_data).loc[idx:idx]
    
    # Get feature values
    X_forecast = features_at_idx[feature_cols]
    
    # Predict
    pred = gb_model.predict(X_forecast)[0]
    ml_forecasts.append(pred)
    
    # Update the forecast_data with prediction for next iteration
    forecast_data.loc[idx, 'value'] = pred
    
    if (i + 1) % 100 == 0:
        print(f"  Generated {i + 1}/{n_forecast} forecasts...")

ml_forecast_df = pd.DataFrame({
    'ML': ml_forecasts
}, index=forecast_period.index)

print(f"\nML forecast generated for {n_forecast} time steps")
print(f"\nFirst 10 forecasts:")
print(ml_forecast_df.head(10))
print(f"\nForecast statistics:")
print(ml_forecast_df.describe())

## 8. Output and Visualization

Display and compare all three forecasts.

In [ ]:
# Combine all forecasts
all_forecasts = pd.concat([arima_forecast_df, ucm_forecast_df, ml_forecast_df], axis=1)

print("All Forecasts Combined:")
print(f"\nFirst 20 forecasts:")
print(all_forecasts.head(20))
print(f"\nLast 20 forecasts:")
print(all_forecasts.tail(20))
print(f"\nForecast statistics:")
print(all_forecasts.describe())

In [ ]:
# Plot all forecasts together
plt.figure(figsize=(15, 8))

# Plot forecasts
plt.plot(all_forecasts.index, all_forecasts['ARIMA'], 
         label='ARIMA', linewidth=2, alpha=0.7)
plt.plot(all_forecasts.index, all_forecasts['UCM'], 
         label='UCM', linewidth=2, alpha=0.7)
plt.plot(all_forecasts.index, all_forecasts['ML'], 
         label='ML (Gradient Boosting)', linewidth=2, alpha=0.7)

plt.title('Forecast Comparison: All Three Models', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('People Count', fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot historical data with forecast transition
# Show last 30 days of history and all forecasts
last_30_days = train_data.last('30D')

plt.figure(figsize=(15, 8))

# Plot last 30 days of history
plt.plot(last_30_days.index, last_30_days['value'], 
         label='Historical Data', linewidth=2, color='black', alpha=0.8)

# Plot forecasts
plt.plot(all_forecasts.index, all_forecasts['ARIMA'], 
         label='ARIMA Forecast', linewidth=2, alpha=0.7, linestyle='--')
plt.plot(all_forecasts.index, all_forecasts['UCM'], 
         label='UCM Forecast', linewidth=2, alpha=0.7, linestyle='--')
plt.plot(all_forecasts.index, all_forecasts['ML'], 
         label='ML Forecast', linewidth=2, alpha=0.7, linestyle='--')

# Add vertical line at forecast start
plt.axvline(x=forecast_period.index[0], color='red', linestyle=':', 
            linewidth=2, label='Forecast Start', alpha=0.5)

plt.title('Historical Data and Forecasts (Last 30 Days + Forecast Period)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('People Count', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot first week of forecasts for detailed view
first_week = all_forecasts.head(7 * 24)

plt.figure(figsize=(15, 8))

plt.plot(first_week.index, first_week['ARIMA'], 
         label='ARIMA', linewidth=2, alpha=0.7, marker='o', markersize=3)
plt.plot(first_week.index, first_week['UCM'], 
         label='UCM', linewidth=2, alpha=0.7, marker='s', markersize=3)
plt.plot(first_week.index, first_week['ML'], 
         label='ML (Gradient Boosting)', linewidth=2, alpha=0.7, marker='^', markersize=3)

plt.title('First Week of Forecasts (Detailed View)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('People Count', fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary

This notebook has successfully:

1. **Loaded** the `student_dataset.csv` with proper datetime parsing
2. **Identified** 1439 missing observations from January 1, 2020 to February 29, 2020
3. **Visualized** the historical time series showing patterns and trends
4. **Implemented three forecasting models:**
   - **ARIMA(5,1,2)**: Classical time series model for autoregressive patterns
   - **UCM**: Unobserved Components Model with local linear trend and 24-hour seasonality
   - **Gradient Boosting**: Machine learning model with engineered features including temporal features, lag features, and rolling statistics
5. **Generated** 1439 hourly forecasts for each model
6. **Visualized** and compared the forecasts

### Key Observations:

- All three models capture different aspects of the time series:
  - ARIMA focuses on autoregressive patterns and trends
  - UCM explicitly models trend and seasonal components
  - ML model leverages complex feature interactions and patterns

- The forecasts can be used to understand expected people counts during the missing period
- Each model has its strengths and can be evaluated based on domain knowledge and expected patterns

In [ ]:
# Save forecasts to CSV for future use (optional)
all_forecasts.to_csv('forecasts_output.csv')
print("Forecasts saved to 'forecasts_output.csv'")